In [133]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

In [134]:

# Cargar los datos desde el archivo Excel
df = pd.read_excel("data/KAKEBO.xlsx", 
                    sheet_name="GASTOS", 
                    engine="openpyxl")

# Filtrar los datos hasta octubre (primeros 10 meses)
df = df.iloc[:10]

df

,MES,EGRESOS
0,1,14057968
1,2,3310580
2,3,3072536
3,4,3455498
4,5,3385952
5,6,3167049
6,7,3337777
7,8,4465156
8,9,5206932
9,10,6291127


In [135]:
df = df.drop(0)#se elimina el primer renglon del dataframe, ya que es enero y representa un valor atípico

df

,MES,EGRESOS
1,2,3310580
2,3,3072536
3,4,3455498
4,5,3385952
5,6,3167049
6,7,3337777
7,8,4465156
8,9,5206932
9,10,6291127


In [136]:
df.mean()

MES        6.000000e+00
EGRESOS    3.965845e+06
dtype: float64

In [137]:
df.describe()

,MES,EGRESOS
count,9.000000,9.000000e+00
mean,6.000000,3.965845e+06
std,2.738613,1.121004e+06
min,2.000000,3.072536e+06
25%,4.000000,3.310580e+06
50%,6.000000,3.385952e+06
75%,8.000000,4.465156e+06
max,10.000000,6.291127e+06


In [138]:
# Crear una variable numérica para representar los meses
df["MES_NUM"] = np.arange(1, len(df) + 1)

# Variables predictoras (X) y variable objetivo (y)
X = df[["MES"]]
y = df["EGRESOS"]


modelo = LinearRegression()# USO DEL MODELO DE REGRESION LINEAL SIMPLE

# Evaluar el modelo con validación cruzada (5 particiones)
scores = cross_val_score(modelo, X, y, scoring="neg_root_mean_squared_error", cv=5)
rmse_promedio = -scores.mean()

# Entrenar el modelo con todos los datos disponibles
modelo.fit(X, y)

# Predecir los valores para noviembre (mes 11) y diciembre (mes 12)
meses_futuros = pd.DataFrame({"MES": [11, 12]})#ajuste de los meses a predecir
predicciones = modelo.predict(meses_futuros)

# Mostrar resultados 
print("Predicciones de gastos proyectadas:")
print(f"Noviembre: ${predicciones[0]:,.2f}")
print(f"Diciembre: ${predicciones[1]:,.2f}")


Predicciones de gastos proyectadas:
Noviembre: $5,657,221.64
Diciembre: $5,995,496.92


In [139]:
#colorear los valores del df segun su intensidad
# se declara la variable 
# se toman el df para colorearlo como se desee
color_table = sns.light_palette("red", as_cmap=True)

# visualizar el df
print("Representando el estilo del DataFrame:")
df.style.background_gradient(cmap=color_table).format(precision=0)

#eliminar esa columna de MES_NUM antes de aplicar el estilo
df_styled = df.drop('MES_NUM', axis=1)
df_styled.style.background_gradient(cmap=color_table).format(precision=0)

Representando el estilo del DataFrame:


,MES,EGRESOS
1,2,3310580
2,3,3072536
3,4,3455498
4,5,3385952
5,6,3167049
6,7,3337777
7,8,4465156
8,9,5206932
9,10,6291127


In [140]:
#lista de meses PAL GRAFICO
meses = ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']


fig = go.Figure()#crear grafica de barras

# Add actual data as bars
fig.add_trace(go.Bar(
    x=df['MES'],
    y=df['EGRESOS'],
    name='Datos Reales',
    marker_color='skyblue'
))

# Add predicted data as bars
fig.add_trace(go.Bar(
    x=meses_futuros['MES'],
    y=predicciones,
    name='Predicciones',
    marker_color='salmon'
))

# Update layout
fig.update_layout(
    title={
        'text': 'GASTOS CASA 2025 (KAKEBO)',
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Mes',
    yaxis_title='Monto Peso en Colombiano (COP)',
    xaxis=dict(
        ticktext=meses,
        tickvals=list(range(1, 13))
    ),
    barmode='group',
    height=600,
    showlegend=True
)
# Add text annotations for each bar
for trace in fig.data:
    fig.add_trace(go.Scatter(
        x=trace.x,
        y=trace.y,
        mode='text',
        text=[f'${y:,.0f}' for y in trace.y],
        textposition='top center',
        showlegend=False
    ))
fig.show()

In [141]:
# Create a DataFrame with actual data and predictions
df_pred = pd.DataFrame({
    'MES': range(2, 13),  # Months 2-12
    'MES_NOMBRE': meses[1:],  # Month names (excluding January)
    'EGRESOS': list(df['EGRESOS']) + list(predicciones),  # Combine actual and predicted values
    'TIPO': ['Real'] * 9 + ['Predicción'] * 2  # Mark which values are real vs predicted
})

# Format EGRESOS values as integers (no decimals)
df_pred['EGRESOS'] = df_pred['EGRESOS'].astype(int)

# Format the EGRESOS column to show values as currency
#df_pred['EGRESOS'] = df_pred['EGRESOS'].round(2)

# Export to CSV
df_pred.to_csv('output/KAKEBO_predicciones.csv', index=False)

# Display the final DataFrame
df_pred

,MES,MES_NOMBRE,EGRESOS,TIPO
0,2,Febrero,3310580,Real
1,3,Marzo,3072536,Real
2,4,Abril,3455498,Real
3,5,Mayo,3385952,Real
4,6,Junio,3167049,Real
5,7,Julio,3337777,Real
6,8,Agosto,4465156,Real
7,9,Septiembre,5206932,Real
8,10,Octubre,6291127,Real
9,11,Noviembre,5657221,Predicción
